# GRPO (Group Relative Policy Optimization)

DeepSeek提出的强化学习算法，通过group内相对比较优化策略。

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
np.random.seed(42)

In [ ]:
class GRPOTrainer:
    def __init__(self, group_size=4, temperature=1.0, advantage_normalization=True):
        self.group_size = group_size
        self.temperature = temperature
        self.advantage_normalization = advantage_normalization
    
    def compute_group_advantages(self, rewards, method='mean_baseline'):
        if method == 'mean_baseline':
            baseline = np.mean(rewards)
        elif method == 'min_baseline':
            baseline = np.min(rewards)
        else:
            baseline = np.median(rewards)
        
        advantages = rewards - baseline
        
        if self.advantage_normalization and len(advantages) > 1:
            advantages = (advantages - np.mean(advantages)) / (np.std(advantages) + 1e-8)
        
        return advantages
    
    def grpo_loss(self, log_probs, rewards):
        advantages = self.compute_group_advantages(rewards)
        advantages = advantages / self.temperature
        policy_loss = -np.mean(log_probs * advantages)
        
        metrics = {
            'loss': policy_loss,
            'reward_mean': np.mean(rewards),
            'reward_std': np.std(rewards),
            'advantage_mean': np.mean(advantages)
        }
        
        return policy_loss, metrics

In [ ]:
# 创建训练器
grpo = GRPOTrainer(group_size=4, temperature=1.0)

# 模拟数据
rewards = np.array([0.8, 0.5, 0.9, 0.3])
log_probs = np.array([-2.5, -3.0, -2.3, -3.5])

# 计算损失
loss, metrics = grpo.grpo_loss(log_probs, rewards)

print(f"损失: {loss:.4f}")
print(f"指标: {metrics}")

In [ ]:
# 可视化group内的相对优势
advantages = grpo.compute_group_advantages(rewards)

fig, ax = plt.subplots(figsize=(10, 6))
x = np.arange(len(rewards))
ax.bar(x, rewards, alpha=0.6, label='奖励')
ax.bar(x, advantages, alpha=0.6, label='相对优势')
ax.axhline(y=0, color='gray', linestyle='--', alpha=0.5)
ax.set_xlabel('响应索引')
ax.set_ylabel('值')
ax.set_title('Group内奖励与相对优势')
ax.legend()
ax.grid(True, alpha=0.3)
plt.show()